# HSCredit 可视化模块 (viz) 完整演示

本 Notebook 演示 hscredit.core.viz 模块的所有可视化功能，共25+个函数：

**1. 基础分箱图表 (binning_plots)**
- `bin_plot`: 特征分箱可视化图
- `corr_plot`: 特征相关性热力图
- `ks_plot`: KS曲线和ROC曲线
- `hist_plot`: 特征分布直方图
- `psi_plot`: PSI稳定性分析图
- `dataframe_plot`: DataFrame表格图
- `distribution_plot`: 样本时间分布图
- `bin_trend_plot`: 特征分箱风险趋势图
- `batch_bin_trend_plot`: 批量特征分箱趋势图
- `overdues_bin_plot`: 多逾期天数分箱图

**2. 模型图表 (model_plots)**
- `plot_weights`: 逻辑回归系数误差图

**3. 金融风控专用图表 (risk_plots)**
- 模型评估: `roc_plot`, `pr_plot`, `lift_plot`, `gain_plot`, `confusion_matrix_plot`, `calibration_plot`
- 评分卡: `score_dist_plot`, `score_bin_plot`
- 风控策略: `threshold_analysis_plot`, `strategy_compare_plot`, `vintage_plot`
- 趋势分析: `feature_importance_plot`, `approval_rate_trend_plot`, `bad_rate_trend_plot`

**输出目录**: `examples/output/viz_demo/`

## 1. 环境准备与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建输出目录
output_dir = 'output/viz_demo'
os.makedirs(output_dir, exist_ok=True)
print(f"输出目录: {output_dir}")

# 加载示例数据
df = pd.read_excel('hscredit.xlsx')
print(f"\n数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
df.head()

In [ ]:
# 数据预处理
if 'target' not in df.columns:
    np.random.seed(42)
    df['target'] = np.random.binomial(1, 0.15, len(df))

# 准备数值特征
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'target']
features_demo = numeric_cols[:6] if len(numeric_cols) >= 6 else numeric_cols

print(f"演示特征: {features_demo}")
print(f"目标分布: \n{df['target'].value_counts()}")

## 2. 基础分箱图表 (binning_plots)

### 2.1 bin_plot - 特征分箱图 (原始数据模式)

In [ ]:
from hscredit.core.viz import bin_plot

if len(features_demo) > 0:
    fig = bin_plot(
        data=df,
        target='target',
        feature=features_demo[0],
        desc=f"特征: {features_demo[0]}",
        n_bins=8,
        method='quantile',
        figsize=(12, 6),
        save=f'{output_dir}/01_bin_plot_raw.png'
    )
    plt.show()
    print(f"✓ 已保存: 01_bin_plot_raw.png")

In [ ]:
# 方式2: 垂直方向 + 自定义分箱
if len(features_demo) > 1:
    feature_vals = df[features_demo[1]].dropna()
    custom_rules = [
        feature_vals.quantile(0),
        feature_vals.quantile(0.25),
        feature_vals.quantile(0.5),
        feature_vals.quantile(0.75),
        feature_vals.quantile(1)
    ]
    
    fig = bin_plot(
        data=df,
        target='target',
        feature=features_demo[1],
        desc=f"特征: {features_demo[1]}",
        rules=custom_rules,
        orientation='vertical',
        figsize=(10, 7),
        save=f'{output_dir}/02_bin_plot_vertical.png'
    )
    plt.show()
    print(f"✓ 已保存: 02_bin_plot_vertical.png")

In [ ]:
# 方式3: 使用分箱统计表（scorecardpipeline模式）
from hscredit.core.metrics import compute_bin_stats
from hscredit.core.binning import OptimalBinning

if len(features_demo) > 0:
    # 创建分箱器
    binner = OptimalBinning(method='quantile', max_n_bins=8, verbose=False)
    X = df[[features_demo[0]]].dropna()
    y = df.loc[X.index, 'target']
    
    binner.fit(X, y)
    bin_indices = binner.transform(X, metric='indices').values.flatten()
    
    # 计算分箱统计
    bin_table = compute_bin_stats(bin_indices, y.values)
    
    # 使用分箱表绘图
    fig = bin_plot(
        data=bin_table,
        desc=f"分箱表模式: {features_demo[0]}",
        figsize=(12, 6),
        save=f'{output_dir}/03_bin_plot_table.png'
    )
    plt.show()
    print(f"✓ 已保存: 03_bin_plot_table.png")

### 2.2 corr_plot - 特征相关性热力图

In [ ]:
from hscredit.core.viz import corr_plot

if len(features_demo) >= 4:
    corr_data = df[features_demo[:5] + ['target']]
    
    # 完整热力图
    fig = corr_plot(corr_data, figure_size=(10, 8), mask=False,
                    save=f'{output_dir}/04_corr_plot_full.png')
    plt.show()
    print(f"✓ 已保存: 04_corr_plot_full.png")
    
    # 下三角热力图
    fig = corr_plot(corr_data, figure_size=(10, 8), mask=True,
                    save=f'{output_dir}/05_corr_plot_masked.png')
    plt.show()
    print(f"✓ 已保存: 05_corr_plot_masked.png")

### 2.3 ks_plot - KS曲线和ROC曲线

In [ ]:
from hscredit.core.viz import ks_plot

np.random.seed(42)
df['score'] = np.random.beta(2, 5, len(df)) * 1000

fig = ks_plot(
    score=df['score'],
    target=df['target'],
    title="模型KS曲线",
    fontsize=14,
    figsize=(14, 6),
    save=f'{output_dir}/06_ks_plot.png'
)
plt.show()
print(f"✓ 已保存: 06_ks_plot.png")

### 2.4 hist_plot - 特征分布直方图

In [ ]:
from hscredit.core.viz import hist_plot

if len(features_demo) > 0:
    fig = hist_plot(
        score=df[features_demo[0]],
        y_true=df['target'],
        figsize=(12, 6),
        bins=30,
        kde=True,
        save=f'{output_dir}/07_hist_plot.png'
    )
    plt.show()
    print(f"✓ 已保存: 07_hist_plot.png")

### 2.5 psi_plot - PSI稳定性分析图

In [ ]:
from hscredit.core.viz import psi_plot

# 创建分箱统计表用于PSI分析
np.random.seed(42)

# 模拟两个时间段的分布
train_scores = np.random.normal(500, 100, 5000)
test_scores = np.random.normal(480, 110, 4000)

# 创建分箱统计
bins = [0, 400, 500, 600, 700, 800, 1000]
bin_labels = ['0-400', '400-500', '500-600', '600-700', '700-800', '800+']

train_bin = pd.cut(train_scores, bins=bins, labels=bin_labels)
test_bin = pd.cut(test_scores, bins=bins, labels=bin_labels)

train_counts = pd.Series(train_bin).value_counts().sort_index()
test_counts = pd.Series(test_bin).value_counts().sort_index()

expected = pd.DataFrame({
    '分箱': bin_labels,
    '样本总数': train_counts.values,
    '样本占比': train_counts.values / train_counts.sum(),
    '坏样本率': np.random.uniform(0.05, 0.2, len(bin_labels))
})

actual = pd.DataFrame({
    '分箱': bin_labels,
    '样本总数': test_counts.values,
    '样本占比': test_counts.values / test_counts.sum(),
    '坏样本率': np.random.uniform(0.05, 0.2, len(bin_labels))
})

fig = psi_plot(
    expected=expected,
    actual=actual,
    labels=['训练集(预期)', '测试集(实际)'],
    desc='评分分布PSI分析',
    save=f'{output_dir}/08_psi_plot.png'
)
plt.show()
print(f"✓ 已保存: 08_psi_plot.png")

### 2.6 dataframe_plot - DataFrame表格图

In [ ]:
from hscredit.core.viz import dataframe_plot

stats_df = pd.DataFrame({
    '特征名': features_demo[:5] if len(features_demo) >= 5 else features_demo + ['feat_4', 'feat_5'],
    'IV值': np.round(np.random.uniform(0.1, 0.8, 5), 3),
    'KS值': np.round(np.random.uniform(0.2, 0.5, 5), 3),
    '缺失率': np.round(np.random.uniform(0, 0.1, 5), 3),
    'Unique数': np.random.randint(10, 1000, 5),
})

fig = dataframe_plot(
    stats_df,
    row_height=0.5,
    font_size=12,
    header_color='#2E86AB',
    save=f'{output_dir}/09_dataframe_plot.png'
)
plt.show()
print(f"✓ 已保存: 09_dataframe_plot.png")

### 2.7 distribution_plot - 样本时间分布图

In [ ]:
from hscredit.core.viz import distribution_plot

# 创建日期数据
np.random.seed(42)
df['apply_date'] = pd.date_range(start='2023-01-01', periods=len(df), freq='D')

fig = distribution_plot(
    df,
    date='apply_date',
    target='target',
    figsize=(14, 6),
    freq='M',
    save=f'{output_dir}/10_distribution_plot.png'
)
plt.show()
print(f"✓ 已保存: 10_distribution_plot.png")

### 2.8 bin_trend_plot - 特征分箱风险趋势图

In [ ]:
from hscredit.core.viz import bin_trend_plot

# 添加月份维度
df['month'] = df['apply_date'].dt.to_period('M').astype(str)

if len(features_demo) > 0:
    fig = bin_trend_plot(
        df=df,
        feature=features_demo[0],
        target='target',
        dimension_cols='month',
        n_bins=6,
        method='quantile',
        orientation='horizontal',
        max_groups=6,
        save=f'{output_dir}/11_bin_trend_plot.png'
    )
    plt.show()
    print(f"✓ 已保存: 11_bin_trend_plot.png")

In [ ]:
# 按日期维度展示
if len(features_demo) > 0:
    fig = bin_trend_plot(
        df=df,
        feature=features_demo[0],
        target='target',
        date_col='apply_date',
        date_freq='M',
        n_bins=5,
        orientation='vertical',
        max_groups=6,
        save=f'{output_dir}/12_bin_trend_plot_date.png'
    )
    plt.show()
    print(f"✓ 已保存: 12_bin_trend_plot_date.png")

### 2.9 batch_bin_trend_plot - 批量特征分箱趋势图

In [ ]:
from hscredit.core.viz import batch_bin_trend_plot

if len(features_demo) >= 3:
    results = batch_bin_trend_plot(
        df=df,
        features=features_demo[:3],
        target='target',
        date_col='apply_date',
        date_freq='M',
        sort_by='iv',
        max_features=3,
        save_dir=output_dir
    )
    print(f"✓ 批量生成完成，共 {len(results)} 个特征")

### 2.10 overdues_bin_plot - 多逾期天数分箱图

In [ ]:
from hscredit.core.viz import overdues_bin_plot

# 模拟逾期数据
np.random.seed(42)
df['dpd7'] = np.random.poisson(3, len(df))
df['dpd15'] = np.random.poisson(5, len(df))
df['dpd30'] = np.random.poisson(8, len(df))

# 方式1: 使用原始数据
if len(features_demo) > 0:
    fig = overdues_bin_plot(
        df=df,
        feature=features_demo[0],
        dpd_cols=['dpd7', 'dpd15', 'dpd30'],
        dpd_thresholds=[1, 1, 1],
        n_bins=5,
        method='quantile',
        max_cols=3,
        save=f'{output_dir}/13_overdues_bin_plot_raw.png'
    )
    plt.show()
    print(f"✓ 已保存: 13_overdues_bin_plot_raw.png")

In [ ]:
# 方式2: 使用分箱表（模拟 feature_bin_stats 输出格式）
from hscredit.core.metrics import compute_bin_stats

if len(features_demo) > 0:
    # 模拟创建多级表头的分箱表
    feature = features_demo[0]
    
    # 为每个DPD创建目标变量
    y_dpd7 = (df['dpd7'] >= 1).astype(int)
    y_dpd15 = (df['dpd15'] >= 1).astype(int)
    y_dpd30 = (df['dpd30'] >= 1).astype(int)
    
    # 分箱
    binner = OptimalBinning(method='quantile', max_n_bins=5, verbose=False)
    X = df[[feature]].dropna()
    
    # 创建多级表头分箱表
    bin_tables = []
    for dpd_name, y_col in [('DPD7', y_dpd7), ('DPD15', y_dpd15), ('DPD30', y_dpd30)]:
        binner.fit(X, y_col.loc[X.index])
        bin_indices = binner.transform(X, metric='indices').values.flatten()
        stats = compute_bin_stats(bin_indices, y_col.loc[X.index].values)
        bin_tables.append((dpd_name, stats))
    
    # 构建多级表头DataFrame
    common_cols = ['分箱', '分箱标签', '样本总数', '样本占比']
    target_cols = ['好样本数', '坏样本数', '坏样本率', '累计好样本占比', '累计坏样本占比', 'Lift', 'WOE值', 'IV值']
    
    multi_cols = [('分箱详情', col) for col in common_cols]
    for dpd_name, _ in bin_tables:
        multi_cols.extend([(dpd_name, col) for col in target_cols])
    
    # 合并数据
    result_data = {}
    base_stats = bin_tables[0][1]
    
    for col in common_cols:
        result_data[('分箱详情', col)] = base_stats[col].values if col in base_stats.columns else range(len(base_stats))
    
    for dpd_name, stats in bin_tables:
        for col in target_cols:
            if col in stats.columns:
                result_data[(dpd_name, col)] = stats[col].values
            else:
                result_data[(dpd_name, col)] = [0] * len(stats)
    
    bin_table_multi = pd.DataFrame(result_data, columns=pd.MultiIndex.from_tuples(multi_cols))
    
    # 使用分箱表绘图
    fig = overdues_bin_plot(
        bin_table=bin_table_multi,
        save=f'{output_dir}/14_overdues_bin_plot_table.png'
    )
    plt.show()
    print(f"✓ 已保存: 14_overdues_bin_plot_table.png")

## 3. 模型图表 (model_plots)

### 3.1 plot_weights - 逻辑回归系数误差图

In [ ]:
from hscredit.core.viz import plot_weights

if len(features_demo) >= 3:
    # 创建模拟的逻辑回归摘要
    coef_data = []
    for feat in features_demo[:5]:
        coef = np.random.uniform(-1, 1)
        std_err = abs(coef) * 0.15 + 0.05
        ci_lower = coef - 1.96 * std_err
        ci_upper = coef + 1.96 * std_err
        coef_data.append({
            'Feature': feat,
            'Coef.': coef,
            'Std.Err': std_err,
            'z': coef / std_err,
            'P>|z|': np.random.uniform(0, 0.2),
            '[0.025': ci_lower,
            '0.975]': ci_upper
        })
    
    # 添加截距
    intercept_coef = -2.5
    intercept_std = 0.2
    coef_data.insert(0, {
        'Feature': 'intercept',
        'Coef.': intercept_coef,
        'Std.Err': intercept_std,
        'z': intercept_coef / intercept_std,
        'P>|z|': 0.001,
        '[0.025': intercept_coef - 1.96 * intercept_std,
        '0.975]': intercept_coef + 1.96 * intercept_std
    })
    
    summary_df = pd.DataFrame(coef_data).set_index('Feature')
    
    fig = plot_weights(
        summary_df,
        figsize=(12, 8),
        fontsize=12,
        save=f'{output_dir}/15_plot_weights.png'
    )
    plt.show()
    print(f"✓ 已保存: 15_plot_weights.png")

## 4. 金融风控专用图表 (risk_plots)

### 4.1 模型评估图表 - ROC, PR, Lift, Gain

In [ ]:
from hscredit.core.viz import roc_plot, pr_plot, lift_plot, gain_plot

# 生成模拟预测概率
np.random.seed(42)
y_true = df['target'].values
y_score_model1 = np.random.beta(2, 5, len(df))
y_score_model2 = np.random.beta(2.5, 4, len(df))

# ROC曲线
fig = roc_plot(y_true, y_score_model1, title="ROC Curve",
               figsize=(8, 8), save=f'{output_dir}/16_roc_plot.png')
plt.show()
print(f"✓ 已保存: 16_roc_plot.png")

# PR曲线
fig = pr_plot(y_true, y_score_model1, title="Precision-Recall Curve",
              figsize=(8, 8), save=f'{output_dir}/17_pr_plot.png')
plt.show()
print(f"✓ 已保存: 17_pr_plot.png")

# Lift图
fig = lift_plot(y_true, y_score_model1, n_bins=10,
                title="Lift Chart", figsize=(10, 6),
                save=f'{output_dir}/18_lift_plot.png')
plt.show()
print(f"✓ 已保存: 18_lift_plot.png")

# Gain图
fig = gain_plot(y_true, y_score_model1, n_bins=10,
                title="Cumulative Gain Chart", figsize=(10, 6),
                save=f'{output_dir}/19_gain_plot.png')
plt.show()
print(f"✓ 已保存: 19_gain_plot.png")

### 4.2 混淆矩阵和校准曲线

In [ ]:
from hscredit.core.viz import confusion_matrix_plot, calibration_plot

y_pred = (y_score_model1 >= 0.3).astype(int)

# 混淆矩阵（计数）
fig = confusion_matrix_plot(y_true, y_pred, normalize=None,
                            title="Confusion Matrix (Count)",
                            figsize=(8, 6),
                            save=f'{output_dir}/20_confusion_matrix_count.png')
plt.show()
print(f"✓ 已保存: 20_confusion_matrix_count.png")

# 混淆矩阵（归一化）
fig = confusion_matrix_plot(y_true, y_pred, normalize='true',
                            title="Confusion Matrix (Normalized)",
                            figsize=(8, 6),
                            save=f'{output_dir}/21_confusion_matrix_norm.png')
plt.show()
print(f"✓ 已保存: 21_confusion_matrix_norm.png")

# 校准曲线
fig = calibration_plot(y_true, y_score_model1, n_bins=10,
                       title="Calibration Curve", figsize=(10, 8),
                       show_histogram=True,
                       save=f'{output_dir}/22_calibration_plot.png')
plt.show()
print(f"✓ 已保存: 22_calibration_plot.png")

### 4.3 评分卡图表 - 评分分布和分箱

In [ ]:
from hscredit.core.viz import score_dist_plot, score_bin_plot

# 创建评分数据
np.random.seed(42)
df['credit_score'] = np.where(
    df['target'] == 0,
    np.random.normal(700, 80, len(df)),
    np.random.normal(550, 100, len(df))
)
df['credit_score'] = np.clip(df['credit_score'], 300, 1000)

# 评分分布图
fig = score_dist_plot(
    df=df,
    score_col='credit_score',
    target_col='target',
    n_bins=30,
    kde=True,
    title="Credit Score Distribution",
    figsize=(12, 6),
    save=f'{output_dir}/23_score_dist_plot.png'
)
plt.show()
print(f"✓ 已保存: 23_score_dist_plot.png")

# 评分分箱图
fig = score_bin_plot(
    df=df,
    score_col='credit_score',
    target_col='target',
    n_bins=10,
    bin_type='quantile',
    title="Score Binning Analysis",
    figsize=(14, 8),
    show_table=True,
    save=f'{output_dir}/24_score_bin_plot.png'
)
plt.show()
print(f"✓ 已保存: 24_score_bin_plot.png")

### 4.4 风控策略图表 - 阈值分析和策略对比

In [ ]:
from hscredit.core.viz import threshold_analysis_plot, strategy_compare_plot

# 阈值分析
fig = threshold_analysis_plot(
    y_true,
    y_score_model1,
    thresholds=np.linspace(0.01, 0.99, 99),
    title="Threshold Analysis",
    figsize=(12, 8),
    metrics=['precision', 'recall', 'f1', 'approval_rate'],
    save=f'{output_dir}/25_threshold_analysis_plot.png'
)
plt.show()
print(f"✓ 已保存: 25_threshold_analysis_plot.png")

In [ ]:
# 策略对比
strategies = [
    {'name': 'Current Strategy', 'approval_rate': 0.75, 'bad_rate': 0.08, 'ks': 0.40, 'auc': 0.72},
    {'name': 'New Strategy A', 'approval_rate': 0.80, 'bad_rate': 0.06, 'ks': 0.50, 'auc': 0.78},
    {'name': 'New Strategy B', 'approval_rate': 0.70, 'bad_rate': 0.04, 'ks': 0.55, 'auc': 0.80},
    {'name': 'Conservative', 'approval_rate': 0.60, 'bad_rate': 0.03, 'ks': 0.58, 'auc': 0.82},
]

fig = strategy_compare_plot(
    strategies=strategies,
    title="Strategy Comparison",
    figsize=(12, 8),
    metrics=['approval_rate', 'bad_rate', 'ks', 'auc'],
    save=f'{output_dir}/26_strategy_compare_plot.png'
)
plt.show()
print(f"✓ 已保存: 26_strategy_compare_plot.png")

### 4.5 Vintage分析和特征重要性

In [ ]:
from hscredit.core.viz import vintage_plot, feature_importance_plot

# Vintage分析数据
np.random.seed(42)
vintage_months = ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06']
mob_values = list(range(1, 13))

vintage_data = []
for vm in vintage_months:
    base_rate = np.random.uniform(0.01, 0.03)
    for mob in mob_values:
        rate = base_rate + mob * 0.003 + np.random.normal(0, 0.002)
        vintage_data.append({
            'vintage': vm,
            'mob': mob,
            'bad_rate': max(0, min(1, rate))
        })

vintage_df = pd.DataFrame(vintage_data)

# Vintage图
fig = vintage_plot(
    df=vintage_df,
    mob_col='mob',
    target_col='bad_rate',
    vintage_col='vintage',
    title="Vintage Analysis",
    figsize=(14, 8),
    max_mob=12,
    show_heatmap=True,
    save=f'{output_dir}/27_vintage_plot.png'
)
plt.show()
print(f"✓ 已保存: 27_vintage_plot.png")

In [ ]:
# 特征重要性图
features = [f'feature_{i}' for i in range(1, 16)]
importance = np.sort(np.random.exponential(0.1, 15))[::-1]

fig = feature_importance_plot(
    features=features,
    importance=importance,
    top_n=10,
    title="Top 10 Feature Importance",
    figsize=(10, 8),
    horizontal=True,
    show_values=True,
    save=f'{output_dir}/28_feature_importance_plot.png'
)
plt.show()
print(f"✓ 已保存: 28_feature_importance_plot.png")

### 4.6 审批通过率和逾期率趋势

In [ ]:
from hscredit.core.viz import approval_rate_trend_plot, bad_rate_trend_plot

# 添加决策结果列
df['is_approved'] = (df['credit_score'] >= 600).astype(int)

# 审批通过率趋势（使用decision_col）
fig = approval_rate_trend_plot(
    df=df,
    date_col='apply_date',
    decision_col='is_approved',
    freq='M',
    title="Approval Rate Trend",
    figsize=(14, 6),
    show_bad_rate=True,
    target_col='target',
    save=f'{output_dir}/29_approval_rate_trend_plot.png'
)
plt.show()
print(f"✓ 已保存: 29_approval_rate_trend_plot.png")

In [ ]:
# 审批通过率趋势（使用score_col + threshold）
fig = approval_rate_trend_plot(
    df=df,
    date_col='apply_date',
    score_col='credit_score',
    threshold=600,
    freq='M',
    title="Approval Rate Trend (Score-based)",
    figsize=(14, 6),
    save=f'{output_dir}/30_approval_rate_trend_score.png'
)
plt.show()
print(f"✓ 已保存: 30_approval_rate_trend_score.png")

In [ ]:
# 添加客户等级用于维度分析
df['customer_grade'] = pd.cut(
    df['credit_score'],
    bins=[0, 500, 600, 700, 800, 1000],
    labels=['E', 'D', 'C', 'B', 'A']
)

# 逾期率趋势（不分维度）
fig = bad_rate_trend_plot(
    df=df,
    date_col='apply_date',
    target_col='target',
    freq='M',
    title="Bad Rate Trend",
    figsize=(14, 6),
    show_sample_count=True,
    save=f'{output_dir}/31_bad_rate_trend_plot.png'
)
plt.show()
print(f"✓ 已保存: 31_bad_rate_trend_plot.png")

In [ ]:
# 逾期率趋势（按客户等级分维度）
fig = bad_rate_trend_plot(
    df=df,
    date_col='apply_date',
    target_col='target',
    dimension_col='customer_grade',
    freq='M',
    title="Bad Rate Trend by Customer Grade",
    figsize=(14, 6),
    show_sample_count=True,
    save=f'{output_dir}/32_bad_rate_trend_by_grade.png'
)
plt.show()
print(f"✓ 已保存: 32_bad_rate_trend_by_grade.png")

## 5. 综合演示 - 模型报告仪表板

In [ ]:
# 创建综合仪表板
fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 子图1: ROC
ax1 = fig.add_subplot(gs[0, 0])
roc_plot(y_true, y_score_model1, ax=ax1, title='ROC Curve')

# 子图2: PR
ax2 = fig.add_subplot(gs[0, 1])
pr_plot(y_true, y_score_model1, ax=ax2, title='PR Curve')

# 子图3: 混淆矩阵
ax3 = fig.add_subplot(gs[0, 2])
confusion_matrix_plot(y_true, y_pred, ax=ax3, title='Confusion Matrix', show_metrics=True)

# 子图4: 评分分布
ax4 = fig.add_subplot(gs[1, 0])
score_dist_plot(df=df, score_col='credit_score', target_col='target', ax=ax4, title='Score Distribution')

# 子图5: Lift
ax5 = fig.add_subplot(gs[1, 1])
lift_plot(y_true, y_score_model1, ax=ax5, title='Lift Chart')

# 子图6: Gain
ax6 = fig.add_subplot(gs[1, 2])
gain_plot(y_true, y_score_model1, ax=ax6, title='Gain Chart')

# 子图7: 校准曲线
ax7 = fig.add_subplot(gs[2, :2])
calibration_plot(y_true, y_score_model1, ax=ax7, title='Calibration Curve')

# 子图8: 特征重要性
ax8 = fig.add_subplot(gs[2, 2])
feature_importance_plot(features=features[:8], importance=importance[:8], ax=ax8, title='Top Features')

fig.suptitle('Model Performance Dashboard', fontsize=18, fontweight='bold', y=0.98)
plt.savefig(f'{output_dir}/33_model_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ 已保存: 33_model_dashboard.png")

## 6. 工具函数演示

In [ ]:
# 演示工具函数的使用
from hscredit.core.viz import setup_axis_style, save_figure, get_or_create_ax, format_bin_label, DEFAULT_COLORS

print("DEFAULT_COLORS:", DEFAULT_COLORS)

# 演示 format_bin_label
test_labels = [
    "[0.0, 100.0)",
    "[100.0, 200.0)",
    "Missing",
    "Special"
]
for label in test_labels:
    formatted = format_bin_label(label, max_len=20)
    print(f"  {label} -> {formatted}")

# 演示 get_or_create_ax
fig, ax = get_or_create_ax(figsize=(8, 6))
ax.plot([1, 2, 3], [1, 4, 2])
ax.set_title('Demo using get_or_create_ax')
setup_axis_style(ax, hide_top_right=True)
plt.savefig(f'{output_dir}/34_utils_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ 已保存: 34_utils_demo.png")

## 7. 演示总结

In [ ]:
import os

output_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.png')])
print("=" * 70)
print(f"viz 模块演示完成！共生成 {len(output_files)} 张图表")
print("=" * 70)

categories = {
    '基础分箱图': [f for f in output_files if any(x in f for x in ['bin_plot', 'corr_plot', 'ks_plot', 'hist_plot', 'psi_plot', 'dataframe_plot', 'distribution_plot'])],
    '趋势分析图': [f for f in output_files if 'trend_plot' in f or 'trend' in f or 'overdues' in f],
    '模型评估图': [f for f in output_files if any(x in f for x in ['roc_plot', 'pr_plot', 'lift_plot', 'gain_plot', 'confusion_matrix', 'calibration'])],
    '评分卡图': [f for f in output_files if 'score' in f and 'plot' in f],
    '风控策略图': [f for f in output_files if any(x in f for x in ['threshold', 'strategy', 'vintage', 'approval_rate', 'bad_rate'])],
    '特征分析图': [f for f in output_files if 'feature_importance' in f or 'plot_weights' in f],
    '综合报告': [f for f in output_files if 'dashboard' in f],
    '工具函数': [f for f in output_files if 'utils' in f]
}

for cat, files in categories.items():
    if files:
        print(f"\n【{cat}】({len(files)}个)")
        for f in files:
            print(f"  - {f}")

print(f"\n{'=' * 70}")
print(f"所有图表已保存到: {output_dir}/")
print("=" * 70)